# 04 - `score` experiment: LightGBM/RandomForest -> stacked MLP -> ensemble
Refactor of notebook cell 15. **That cell never ran** (no outputs), uses a different file
(`Copy of output_mileage_mtd_scored(1).csv`) and is unrelated to the FinalDecision / Daily_Km models. No results exist for it.

Set `SCORE_CSV` to your own file. Without one, the notebook runs on a randomly generated **toy** frame just to exercise the code.

In [ ]:
import logging, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")

import numpy as np
import pandas as pd
from src.experiments.score_stacking import ScoreExperimentConfig, run_score_experiment

SCORE_CSV = None          # e.g. ROOT / "data/raw/output_mileage_mtd_scored.csv"
TARGET = "score"
USE_MLP = True            # needs PyTorch; set False for the tree stage only

if SCORE_CSV is not None:
    df = pd.read_csv(SCORE_CSV)
else:
    rng = np.random.default_rng(0)
    n = 800
    df = pd.DataFrame({"a": rng.normal(size=n), "b": rng.normal(size=n), "c": rng.choice(["x", "y", "z"], n)})
    df[TARGET] = 2 * df["a"] - df["b"] + df["c"].map({"x": 0, "y": 1, "z": 2}) + rng.normal(0, 0.3, n)
    print("TOY data: results below mean nothing about the real score file")
df.head()

In [ ]:
config = ScoreExperimentConfig(target=TARGET, use_mlp=USE_MLP, n_folds=3 if SCORE_CSV is None else 5)
report = run_score_experiment(df, config, output_dir=ROOT / "outputs" / "latest" / "score_experiment")
pd.DataFrame(report["metrics"]).T[["rmse", "mae", "r2"]]

Reading the table: `tree_oof_train` is out-of-fold on the training rows; `*_test` is the honest number. MLP and ensemble *validation* scores are optimistic because validation drives MLP early stopping.
If `score` is computed from other columns in the file, a high R2 only shows the model re-learned that formula.

In [ ]:
pd.Series(report["top_feature_importance"]).head(15)